In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lag, months_between, desc, expr, instr, when, regexp_replace, to_timestamp, rank, least, split, concat_ws, regexp_extract, lit, monotonically_increasing_id, collect_list, element_at, concat, to_date, date_format, isnan
from pyspark.sql.window import Window

from IPython.display import display

In [ ]:
spark = SparkSession.builder.appName("Basic Transformations").getOrCreate()

<h2>I. Tiền xử lý dữ liệu</h2>

<p2>- Thay đổi, chọn lọc, thêm các thuộc tính<br>- Biến đổi kiểu dữ liệu</p2>

<h3>1. FBref tables</h3>

In [ ]:
links = ["gs://football_data_etl/football_data_extracted/fbref/ENG_fbref_19_24/",
         "gs://football_data_etl/football_data_extracted/fbref/ENG_fbref_2018/",
         "gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_18_23/",
         "gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_24/",
         "gs://football_data_etl/football_data_extracted/fbref/GER_fbref_18_19/",
         "gs://football_data_etl/football_data_extracted/fbref/GER_fbref_20_24/",
         "gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_18_20/",
         "gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_21_24/"]

Advanced_match_stats

In [ ]:
advanced_match_stats = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/advanced_match_stats.csv",
                                 header=True, inferSchema=True)
for i in links:
    temp = spark.read.csv(i+"advanced_match_stats.csv",
                                 header=True, inferSchema=True)
    advanced_match_stats = advanced_match_stats.union(temp)


In [ ]:
advanced_match_stats.count()

25416

In [ ]:
advanced_match_stats.show(5)

+-------+----------+--------------------+-------------+--------------+----------+-------+--------------------+-----------------+--------------+----------+--------------+----------+-------+--------------------+-----------------+--------------+--------------------+-------------+---------+-----------+---+---+---+---+-----+---+---+----+----+-------+---+---+------+-----------+-------------+------------+-------+-------+----------+----------+------------------+-----------+---------------+------------+------------+-------------+
| League|Match_Date|           Matchweek|    Home_Team|Home_Formation|Home_Score|Home_xG|          Home_Goals|Home_Yellow_Cards|Home_Red_Cards| Away_Team|Away_Formation|Away_Score|Away_xG|          Away_Goals|Away_Yellow_Cards|Away_Red_Cards|            Game_URL|         Team|Home_Away|Player_Href|Min|Gls|Ast| PK|PKatt| Sh|SoT|CrdY|CrdR|Touches|Tkl|Int|Blocks|xG_Expected|npxG_Expected|xAG_Expected|SCA_SCA|GCA_SCA|Cmp_Passes|Att_Passes|Cmp_percent_Passes|PrgP_Passes|Car

In [ ]:
advanced_match_stats.filter(isnan("Match_Date")).show()

+------+----------+---------+---------+--------------+----------+-------+----------+-----------------+--------------+---------+--------------+----------+-------+----------+-----------------+--------------+--------+----+---------+-----------+---+---+---+---+-----+---+---+----+----+-------+---+---+------+-----------+-------------+------------+-------+-------+----------+----------+------------------+-----------+---------------+------------+------------+-------------+
|League|Match_Date|Matchweek|Home_Team|Home_Formation|Home_Score|Home_xG|Home_Goals|Home_Yellow_Cards|Home_Red_Cards|Away_Team|Away_Formation|Away_Score|Away_xG|Away_Goals|Away_Yellow_Cards|Away_Red_Cards|Game_URL|Team|Home_Away|Player_Href|Min|Gls|Ast| PK|PKatt| Sh|SoT|CrdY|CrdR|Touches|Tkl|Int|Blocks|xG_Expected|npxG_Expected|xAG_Expected|SCA_SCA|GCA_SCA|Cmp_Passes|Att_Passes|Cmp_percent_Passes|PrgP_Passes|Carries_Carries|PrgC_Carries|Att_Take_Ons|Succ_Take_Ons|
+------+----------+---------+---------+--------------+--------

Match_lineups

In [ ]:
match_lineups = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/match_lineups.csv",
                                 header=True, inferSchema=True)
for i in links:
    temp = spark.read.csv(i+"match_lineups.csv",
                                 header=True, inferSchema=True)
    match_lineups = match_lineups.union(temp)


In [ ]:
match_lineups.count()

507903

In [ ]:
match_lineups.show(5)

+----------+-------------+---------+---------+----------+--------------------+--------+--------------------+------+---+------+---+---+---+----+----+--------------------+
|  Matchday|         Team|Home_Away|Formation|Player_Num|         Player_Name|Starting|           PlayerURL|Nation|Pos|   Age|Min|Gls|Ast|CrdY|CrdR|            MatchURL|
+----------+-------------+---------+---------+----------+--------------------+--------+--------------------+------+---+------+---+---+---+----+----+--------------------+
|11/24/2017|Saint-Étienne|     Home|  4-2-3-1|        16|    Stéphane Ruffier|   Pitch|https://fbref.com...|   FRA| GK|31-058| 90|  0|  0|   0|   0|https://fbref.com...|
|11/24/2017|Saint-Étienne|     Home|  4-2-3-1|         2|Kévin Théophile-C...|   Pitch|https://fbref.com...|   FRA| CB|28-027| 90|  0|  0|   0|   0|https://fbref.com...|
|11/24/2017|Saint-Étienne|     Home|  4-2-3-1|         5|       Vincent Pajot|   Pitch|https://fbref.com...|   FRA| DM|27-097| 85|  0|  0|   1|   0|ht

In [ ]:
match_lineups.filter(isnan("Matchday")).show()

+--------+----+---------+---------+----------+-----------+--------+---------+------+---+---+---+---+---+----+----+--------+
|Matchday|Team|Home_Away|Formation|Player_Num|Player_Name|Starting|PlayerURL|Nation|Pos|Age|Min|Gls|Ast|CrdY|CrdR|MatchURL|
+--------+----+---------+---------+----------+-----------+--------+---------+------+---+---+---+---+---+----+----+--------+
+--------+----+---------+---------+----------+-----------+--------+---------+------+---+---+---+---+---+----+----+--------+



Match_results

In [ ]:
match_results = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/match_results.csv",
                                 header=True, inferSchema=True)
for i in links:
    temp = spark.read.csv(i+"match_results.csv",
                                 header=True, inferSchema=True)
    match_results = match_results.union(temp)


In [ ]:
match_results.count()

12708

In [ ]:
match_results.show(5)

+----------------+------+-------+---------------+-------+---+---+--------+-------------------+-------------+---------+-------+----------+---------+-------+----------+--------------------+------------------+-----+--------------------+
|Competition_Name|Gender|Country|Season_End_Year|  Round| Wk|Day|    Date|               Time|         Home|HomeGoals|Home_xG|      Away|AwayGoals|Away_xG|Attendance|               Venue|           Referee|Notes|            MatchURL|
+----------------+------+-------+---------------+-------+---+---+--------+-------------------+-------------+---------+-------+----------+---------+-------+----------+--------------------+------------------+-----+--------------------+
|         Ligue 1|     M|    FRA|           2018|Ligue 1|  1|Fri|8/4/2017|2024-07-03 20:45:00|       Monaco|        3|      2|  Toulouse|        2|    0.3|     13572|      Stade Louis II|    Clément Turpin| NULL|https://fbref.com...|
|         Ligue 1|     M|    FRA|           2018|Ligue 1|  1|Sat

In [ ]:
match_results.filter(isnan("Date")).show()

+----------------+------+-------+---------------+-----+---+---+----+----+----+---------+-------+----+---------+-------+----------+-----+-------+-----+--------+
|Competition_Name|Gender|Country|Season_End_Year|Round| Wk|Day|Date|Time|Home|HomeGoals|Home_xG|Away|AwayGoals|Away_xG|Attendance|Venue|Referee|Notes|MatchURL|
+----------------+------+-------+---------------+-----+---+---+----+----+----+---------+-------+----+---------+-------+----------+-----+-------+-----+--------+
+----------------+------+-------+---------------+-----+---+---+----+----+----+---------+-------+----+---------+-------+----------+-----+-------+-----+--------+



Match_stats

In [ ]:
match_stats = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/match_stats.csv",
                                 header=True, inferSchema=True)
for i in links:
    temp = spark.read.csv(i+"match_stats.csv",
                                 header=True, inferSchema=True)
    match_stats = match_stats.union(temp)


In [ ]:
match_stats.count()

12621

In [ ]:
match_stats.show(5)

+-------+----------+--------------------+-------------+--------------+---------------+-----------------+---------+---------------------+----------+------------+------------+------------+------------+------------------+----------------+---------------+-------------+---------------+--------------+---------------+-----------------+--------------+-------------+--------------+---------------+-----------------+---------+---------------------+----------+------------+------------+------------+------------+------------------+----------------+---------------+-------------+---------------+--------------+---------------+-----------------+--------------+--------------------+
| League|Match_Date|           Matchweek|    Home_Team|Home_Formation|Home_Possession|Home_Success_Pass|Home_Pass|Home_Passing_Accuracy|Home_Fouls|Home_Corners|Home_Crosses|Home_Touches|Home_Tackles|Home_Interceptions|Home_Aerials_Won|Home_Clearances|Home_Offsides|Home_Goal_Kicks|Home_Throw_Ins|Home_Long_Balls|Home_Yellow_Cards

In [ ]:
match_stats.filter(isnan("Match_Date")).show()

+------+----------+---------+---------+--------------+---------------+-----------------+---------+---------------------+----------+------------+------------+------------+------------+------------------+----------------+---------------+-------------+---------------+--------------+---------------+-----------------+--------------+---------+--------------+---------------+-----------------+---------+---------------------+----------+------------+------------+------------+------------+------------------+----------------+---------------+-------------+---------------+--------------+---------------+-----------------+--------------+--------+
|League|Match_Date|Matchweek|Home_Team|Home_Formation|Home_Possession|Home_Success_Pass|Home_Pass|Home_Passing_Accuracy|Home_Fouls|Home_Corners|Home_Crosses|Home_Touches|Home_Tackles|Home_Interceptions|Home_Aerials_Won|Home_Clearances|Home_Offsides|Home_Goal_Kicks|Home_Throw_Ins|Home_Long_Balls|Home_Yellow_Cards|Home_Red_Cards|Away_Team|Away_Formation|Away_Po

Match_summary

In [ ]:
match_summary = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/match_summary.csv",
                                 header=True, inferSchema=True)
for i in links:
    temp = spark.read.csv(i+"match_summary.csv",
                                 header=True, inferSchema=True)
    match_summary = match_summary.union(temp)


In [ ]:
match_summary.count()

182198

In [ ]:
match_summary.show(5)

+-------+----------+--------------------+-------------+--------------+----------+-------+--------------------+-----------------+--------------+----------+--------------+----------+-------+--------------------+-----------------+--------------+--------------------+-------------+---------+----------+-------+----------+-----------+--------------------+-------------------+--------------+
| League|Match_Date|           Matchweek|    Home_Team|Home_Formation|Home_Score|Home_xG|          Home_Goals|Home_Yellow_Cards|Home_Red_Cards| Away_Team|Away_Formation|Away_Score|Away_xG|          Away_Goals|Away_Yellow_Cards|Away_Red_Cards|            Game_URL|         Team|Home_Away|Event_Time|Is_Pens|Event_Half| Event_Type|       Event_Players|  Score_Progression|Penalty_Number|
+-------+----------+--------------------+-------------+--------------+----------+-------+--------------------+-----------------+--------------+----------+--------------+----------+-------+--------------------+-----------------+-

In [ ]:
match_summary.filter(isnan("Match_Date")).show()

+------+----------+---------+---------+--------------+----------+-------+----------+-----------------+--------------+---------+--------------+----------+-------+----------+-----------------+--------------+--------+----+---------+----------+-------+----------+----------+-------------+-----------------+--------------+
|League|Match_Date|Matchweek|Home_Team|Home_Formation|Home_Score|Home_xG|Home_Goals|Home_Yellow_Cards|Home_Red_Cards|Away_Team|Away_Formation|Away_Score|Away_xG|Away_Goals|Away_Yellow_Cards|Away_Red_Cards|Game_URL|Team|Home_Away|Event_Time|Is_Pens|Event_Half|Event_Type|Event_Players|Score_Progression|Penalty_Number|
+------+----------+---------+---------+--------------+----------+-------+----------+-----------------+--------------+---------+--------------+----------+-------+----------+-----------------+--------------+--------+----+---------+----------+-------+----------+----------+-------------+-----------------+--------------+
+------+----------+---------+---------+-------

Match_players_stats

In [ ]:
#ligue1
ligue1_2018 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/player_stats_ligue1_2018.csv",
                                 header=True, inferSchema=True)
ligue1_2019 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/player_stats_ligue1_2019.csv",
                                 header=True, inferSchema=True)
ligue1_2020 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/player_stats_ligue1_2020.csv",
                                 header=True, inferSchema=True)
ligue1_2021 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/player_stats_ligue1_2021.csv",
                                 header=True, inferSchema=True)
ligue1_2022 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/player_stats_ligue1_2022.csv",
                                 header=True, inferSchema=True)
ligue1_2023 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/player_stats_ligue1_2023.csv",
                                 header=True, inferSchema=True)
ligue1_2024 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/FRA_fbref/player_stats_ligue1_2024.csv",
                                 header=True, inferSchema=True)
match_players_stats1 = ligue1_2018.union(ligue1_2019).union(ligue1_2020).union(ligue1_2021).union(ligue1_2022)
match_players_stats1 = match_players_stats1.union(ligue1_2023).union(ligue1_2024)

#premier_league
premier_2018 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ENG_fbref_2018/player_stats_premier_2018.csv",
                                 header=True, inferSchema=True)
premier_2019 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbrefENG_fbref_19_24/player_stats_premier_2019.csv",
                                 header=True, inferSchema=True)
premier_2020 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ENG_fbref_19_24/player_stats_premier_2020.csv",
                                 header=True, inferSchema=True)
premier_2021 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ENG_fbref_19_24/player_stats_premier_2021.csv",
                                 header=True, inferSchema=True)
premier_2022 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ENG_fbref_19_24/player_stats_premier_2022.csv",
                                 header=True, inferSchema=True)
premier_2023 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ENG_fbref_19_24/player_stats_premier_2023.csv",
                                 header=True, inferSchema=True)
premier_2024 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ENG_fbref_19_24/player_stats_premier_2024.csv",
                                 header=True, inferSchema=True)
match_players_stats2 = premier_2018.union(premier_2019).union(premier_2020).union(premier_2021).union(premier_2022)
match_players_stats2 = match_players_stats2.union(premier_2023).union(premier_2024)

#laliga
laliga_2018 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_18_23/player_stats_laliga_2018.csv",
                                 header=True, inferSchema=True)
laliga_2019 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_18_23/player_stats_laliga_2019.csv",
                                 header=True, inferSchema=True)
laliga_2020 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_18_23/player_stats_laliga_2020.csv",
                                 header=True, inferSchema=True)
laliga_2021 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_18_23/player_stats_laliga_2021.csv",
                                 header=True, inferSchema=True)
laliga_2022 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_18_23/player_stats_laliga_2022.csv",
                                 header=True, inferSchema=True)
laliga_2023 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_18_23/player_stats_laliga_2023.csv",
                                 header=True, inferSchema=True)
laliga_2024 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ESP_fbref_24/player_stats_laliga_2024.csv",
                                 header=True, inferSchema=True)
match_players_stats3 = laliga_2018.union(laliga_2019).union(laliga_2020).union(laliga_2021).union(laliga_2022)
match_players_stats3 = match_players_stats3.union(laliga_2023).union(laliga_2024)

#bundesliga
bundesliga_2018 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/GER_fbref_18_19/player_stats_bundesliga_2018.csv",
                                 header=True, inferSchema=True)
bundesliga_2019 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/GER_fbref_18_19/player_stats_bundesliga_2019.csv",
                                 header=True, inferSchema=True)
bundesliga_2020 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/GER_fbref_20_24/player_stats_bundesliga_2020.csv",
                                 header=True, inferSchema=True)
bundesliga_2021 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/GER_fbref_20_24/player_stats_bundesliga_2021.csv",
                                 header=True, inferSchema=True)
bundesliga_2022 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/GER_fbref_20_24/player_stats_bundesliga_2022.csv",
                                 header=True, inferSchema=True)
bundesliga_2023 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/GER_fbref_20_24/player_stats_bundesliga_2023.csv",
                                 header=True, inferSchema=True)
bundesliga_2024 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/GER_fbref_20_24/player_stats_bundesliga_2024.csv",
                                 header=True, inferSchema=True)
match_players_stats4 = bundesliga_2018.union(bundesliga_2019).union(bundesliga_2020).union(bundesliga_2021).union(bundesliga_2022)
match_players_stats4 = match_players_stats4.union(bundesliga_2023).union(bundesliga_2024)

#serie_a
serie_a_2018 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_18_20/player_stats_seriea_2018.csv",
                                 header=True, inferSchema=True)
serie_a_2019 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_18_20/player_stats_seriea_2019.csv",
                                 header=True, inferSchema=True)
serie_a_2020 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_18_20/player_stats_seriea_2020.csv",
                                 header=True, inferSchema=True)
serie_a_2021 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_21_24/player_stats_seriea_2021.csv",
                                 header=True, inferSchema=True)
serie_a_2022 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_21_24/player_stats_seriea_2022.csv",
                                 header=True, inferSchema=True)
serie_a_2023 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_21_24/player_stats_seriea_2023.csv",
                                 header=True, inferSchema=True)
serie_a_2024 = spark.read.csv("gs://football_data_etl/football_data_extracted/fbref/ITA_fbref_21_24/player_stats_seriea_2024.csv",
                                 header=True, inferSchema=True)
match_players_stats5 = serie_a_2018.union(serie_a_2019).union(serie_a_2020).union(serie_a_2021).union(serie_a_2022)
match_players_stats5 = match_players_stats5.union(serie_a_2023).union(serie_a_2024)

#all
match_players_stats = match_players_stats1.union(match_players_stats2).union(match_players_stats3)
match_players_stats = match_players_stats.union(match_players_stats4).union(match_players_stats5)


In [ ]:
match_players_stats.count()

369519

In [ ]:
match_players_stats.show(5)

+-------+----------+--------------------+---------+--------------+----------+-------+--------------------+-----------------+--------------+---------+--------------+----------+-------+--------------------+-----------------+--------------+--------------------+------+---------+-------------------+--------------------+----------+------+-----+------+---+---+---+---+-----+---+---+----+----+-------+---+---+------+-----------+-------------+------------+-------+-------+----------+----------+------------------+-----------+---------------+------------+------------+-------------+
| League|Match_Date|           Matchweek|Home_Team|Home_Formation|Home_Score|Home_xG|          Home_Goals|Home_Yellow_Cards|Home_Red_Cards|Away_Team|Away_Formation|Away_Score|Away_xG|          Away_Goals|Away_Yellow_Cards|Away_Red_Cards|            Game_URL|  Team|Home_Away|             Player|         Player_Href|Player_Num|Nation|  Pos|   Age|Min|Gls|Ast| PK|PKatt| Sh|SoT|CrdY|CrdR|Touches|Tkl|Int|Blocks|xG_Expected|npx

In [ ]:
match_players_stats.filter(isnan("Match_Date")).show()

+------+----------+---------+---------+--------------+----------+-------+----------+-----------------+--------------+---------+--------------+----------+-------+----------+-----------------+--------------+--------+----+---------+------+-----------+----------+------+---+---+---+---+---+---+-----+---+---+----+----+-------+---+---+------+-----------+-------------+------------+-------+-------+----------+----------+------------------+-----------+---------------+------------+------------+-------------+
|League|Match_Date|Matchweek|Home_Team|Home_Formation|Home_Score|Home_xG|Home_Goals|Home_Yellow_Cards|Home_Red_Cards|Away_Team|Away_Formation|Away_Score|Away_xG|Away_Goals|Away_Yellow_Cards|Away_Red_Cards|Game_URL|Team|Home_Away|Player|Player_Href|Player_Num|Nation|Pos|Age|Min|Gls|Ast| PK|PKatt| Sh|SoT|CrdY|CrdR|Touches|Tkl|Int|Blocks|xG_Expected|npxG_Expected|xAG_Expected|SCA_SCA|GCA_SCA|Cmp_Passes|Att_Passes|Cmp_percent_Passes|PrgP_Passes|Carries_Carries|PrgC_Carries|Att_Take_Ons|Succ_Take_O

FBref_MatchInfos table

In [ ]:
fbref_matchinfos = match_results

fbref_matchinfos = fbref_matchinfos.withColumn("Match_Id", expr("substring(MatchURL, 30, 8)"))

In [ ]:
fbref_matchinfos = fbref_matchinfos.select(['Match_Id','Competition_Name','Season_End_Year','Wk','Home','Away',
                                            'Date','Day','Time','Attendance','Venue','Referee','MatchURL'])

In [ ]:
new_column_names = ['Match_Id', 'League', 'Season', 'Match_Week',
                    'Home_Team', 'Away_Team', 'Match_Date', 'Venue_Day', 'Venue_Time',
                    'Attendance', 'Stadium', 'Officials', 'Link']

for i, col_name in enumerate(fbref_matchinfos.columns):
    fbref_matchinfos = fbref_matchinfos.withColumnRenamed(col_name, new_column_names[i])

In [ ]:
#fbref_matchinfos = fbref_matchinfos.withColumn('Match_Date', to_date('Match_Date', 'M/d/yyyy'))

#match_results = match_results.withColumn("Date", to_date(col("Date"), "M/d/yyyy"))
#match_results = match_results.withColumn("Date", date_format(col("Date"), "yyyy-MM-dd"))

fbref_matchinfos = fbref_matchinfos.withColumn('Match_Date', when(col('Match_Date').contains('/'), date_format(to_date(col("Match_Date"), "M/d/yyyy"), "yyyy-MM-dd")).otherwise(col('Match_Date')))

In [ ]:
#fbref_matchinfos = fbref_matchinfos.withColumn('Match_Date', to_timestamp(col('Match_Date'), 'yyyy-MM-dd'))
fbref_matchinfos.printSchema()

root
 |-- Match_Id: string (nullable = true)
 |-- League: string (nullable = true)
 |-- Season: integer (nullable = true)
 |-- Match_Week: integer (nullable = true)
 |-- Home_Team: string (nullable = true)
 |-- Away_Team: string (nullable = true)
 |-- Match_Date: string (nullable = true)
 |-- Venue_Day: string (nullable = true)
 |-- Venue_Time: timestamp (nullable = true)
 |-- Attendance: string (nullable = true)
 |-- Stadium: string (nullable = true)
 |-- Officials: string (nullable = true)
 |-- Link: string (nullable = true)



In [ ]:
display(fbref_matchinfos.limit(5).toPandas())

,Match_Id,League,Season,Match_Week,Home_Team,Away_Team,Match_Date,Venue_Day,Venue_Time,Attendance,Stadium,Officials,Link
0,a68e623d,Ligue 1,2018,1,Monaco,Toulouse,2017-08-04,Fri,2024-07-03 20:45:00,13572,Stade Louis II,Clément Turpin,https://fbref.com/en/matches/a68e623d/Monaco-T...
1,37f2c25f,Ligue 1,2018,1,Paris S-G,Amiens,2017-08-05,Sat,2024-07-03 17:15:00,46898,Parc des Princes,Mikael Lesage,https://fbref.com/en/matches/37f2c25f/Paris-Sa...
2,4d28b63b,Ligue 1,2018,1,Lyon,Strasbourg,2017-08-05,Sat,2024-07-03 20:00:00,1979,Groupama Stadium,Ruddy Buquet,https://fbref.com/en/matches/4d28b63b/Lyon-Str...
3,68b9eea2,Ligue 1,2018,1,Saint-Étienne,Nice,2017-08-05,Sat,2024-07-03 20:00:00,25879,Stade Geoffroy-Guichard,François Letexier,https://fbref.com/en/matches/68b9eea2/Saint-Et...
4,7d2eb66d,Ligue 1,2018,1,Metz,Guingamp,2017-08-05,Sat,2024-07-03 20:00:00,14595,Stade Saint-Symphorien,Jérôme Miguelgorry,https://fbref.com/en/matches/7d2eb66d/Metz-Gui...


<p3>FBref_MatchSquad table</p3>

In [ ]:
fbref_matchsquad = match_lineups

fbref_matchsquad = fbref_matchsquad.withColumn("Match_Id", expr("substring(MatchURL, 30, 8)"))
fbref_matchsquad = fbref_matchsquad.withColumn("Is_Home_Team", when(col("Home_Away") == "Home", "Yes")
                                               .when(col("Home_Away") == "Away", "No").otherwise("Unknown"))
fbref_matchsquad = fbref_matchsquad.withColumn("Is_Sub", when(col("Starting") == "Pitch", "No")
                                               .when(col("Starting") == "Bench", "Yes").otherwise("Unknown"))

In [ ]:
fbref_matchsquad = fbref_matchsquad.select(['Match_Id', 'Team', 'Is_Home_Team', 'Player_Name', 'Player_Num',
                                            'Is_Sub'])

In [ ]:
new_column_names = ['Match_Id', 'Team', 'Is_Home_Team', 'Player_Name',
                    'Player_Kitnum', 'Is_Sub']

for i, col_name in enumerate(fbref_matchsquad.columns):
    fbref_matchsquad = fbref_matchsquad.withColumnRenamed(col_name, new_column_names[i])

In [ ]:
fbref_matchsquad.printSchema()

root
 |-- Match_Id: string (nullable = true)
 |-- Team: string (nullable = true)
 |-- Is_Home_Team: string (nullable = false)
 |-- Player_Name: string (nullable = true)
 |-- Player_Kitnum: integer (nullable = true)
 |-- Is_Sub: string (nullable = false)



In [ ]:
display(fbref_matchsquad.limit(5).toPandas())

,Match_Id,Team,Is_Home_Team,Player_Name,Player_Kitnum,Is_Sub
0,39b738ac,Saint-Étienne,Yes,Stéphane Ruffier,16,No
1,39b738ac,Saint-Étienne,Yes,Kévin Théophile-Catherine,2,No
2,39b738ac,Saint-Étienne,Yes,Vincent Pajot,5,No
3,39b738ac,Saint-Étienne,Yes,Assane Dioussé,8,No
4,39b738ac,Saint-Étienne,Yes,Jonathan Bamba,14,No


<p3>FBref_MatchGoals table</p3>

In [ ]:
fbref_matchgoals = match_summary

fbref_matchgoals = fbref_matchgoals.withColumn("Match_Id", expr("substring(Game_URL, 30, 8)"))
fbref_matchgoals = fbref_matchgoals.withColumn("Is_Home_Team", when(col("Home_Away") == "Home", "Yes")
                                               .when(col("Home_Away") == "Away", "No").otherwise("Unknown"))

fbref_matchgoals = fbref_matchgoals.withColumn("Type_Of_Goal", when(col("Event_Type") == "Goal", "Normal")
                                               .when(col("Event_Type") == "Own Goal", "OG")
                                               .when(col("Event_Type") == "Penalty", "Penalty").otherwise("Unknown"))

fbref_matchgoals = fbref_matchgoals.withColumn("Player_Name", when(col("Event_Type") == "Own Goal",
                                                                   regexp_extract(col("Event_Players"), "(.*?)Own", 1))
                                               .when((col("Event_Type") == "Penalty")
                                                     & (col("Event_Players").like("%Penalty Kick%")),
                                                                   regexp_extract(col("Event_Players"), "(.*?)Penalty Kick", 1))
                                               .when(col("Event_Type") == "Goal", regexp_replace(col("Event_Players"), " Assist.*$", ""))
                                               .otherwise("Unknown"))

fbref_matchgoals = fbref_matchgoals.withColumn("Minute", when((col("Event_Half") == 1) & (col("Event_Time") > 45),
                                                              concat_ws("+", lit("45"), (col("Event_Time")-45).cast("string"))).
                                               when((col("Event_Half") == 2) & (col("Event_Time") > 90),
                                                              concat_ws("+", lit("90"), (col("Event_Time")-90).cast("string")))
                                               .otherwise(col("Event_Time")))

In [ ]:
fbref_matchgoals = fbref_matchgoals.select(['Match_Id', 'Team', 'Minute', 'Player_Name',
                                              'Type_Of_Goal', 'Is_Home_Team'])

In [ ]:
fbref_matchgoals.printSchema()

root
 |-- Match_Id: string (nullable = true)
 |-- Team: string (nullable = true)
 |-- Minute: string (nullable = true)
 |-- Player_Name: string (nullable = true)
 |-- Type_Of_Goal: string (nullable = false)
 |-- Is_Home_Team: string (nullable = false)



In [ ]:
display(fbref_matchgoals.limit(5).toPandas())

,Match_Id,Team,Minute,Player_Name,Type_Of_Goal,Is_Home_Team
0,39b738ac,Saint-Étienne,10,Unknown,Unknown,Yes
1,39b738ac,Strasbourg,32,Unknown,Unknown,No
2,39b738ac,Saint-Étienne,41,Hernani,Normal,Yes
3,39b738ac,Saint-Étienne,43,Unknown,Unknown,Yes
4,39b738ac,Strasbourg,44,Jean-Eudes Aholou,Normal,No


<p3>FBref_MatchStats table</p3>

In [ ]:
fbref_matchstats = advanced_match_stats.join(match_stats, on="Game_URL")

In [ ]:
fbref_matchstats = fbref_matchstats.withColumn("Match_Id", expr("substring(Game_URL, 30, 8)"))
fbref_matchstats = fbref_matchstats.withColumn("Is_Home_Team", when(col("Home_Away") == "Home", "Yes")
                                               .when(col("Home_Away") == "Away", "No").otherwise("Unknown"))

fbref_matchstats = fbref_matchstats.withColumn("Manager", lit("Unknown"))
fbref_matchstats = fbref_matchstats.withColumn("Captain", lit("Unknown"))
fbref_matchstats = fbref_matchstats.withColumn("Total_Players_Stats", lit("Unknown"))
fbref_matchstats = fbref_matchstats.withColumn("Minutes", col("Min"))

fbref_matchstats = fbref_matchstats.withColumn("PK_Att", col("PKatt"))
fbref_matchstats = fbref_matchstats.withColumn("xG", col("xG_Expected"))
fbref_matchstats = fbref_matchstats.withColumn("npxG", col("npxG_Expected"))
fbref_matchstats = fbref_matchstats.withColumn("xAG", col("xAG_Expected"))
fbref_matchstats = fbref_matchstats.withColumn("SCA", col("SCA_SCA"))
fbref_matchstats = fbref_matchstats.withColumn("GCA", col("GCA_SCA"))

fbref_matchstats = fbref_matchstats.withColumn("Passes_Cmp", col("Cmp_Passes"))
fbref_matchstats = fbref_matchstats.withColumn("Passes_Att", col("Att_Passes"))
fbref_matchstats = fbref_matchstats.withColumn("Passes_Cmp_Percentage", col("Cmp_percent_Passes"))
fbref_matchstats = fbref_matchstats.withColumn("Passes_PrgP", col("PrgP_Passes"))
fbref_matchstats = fbref_matchstats.withColumn("Carries", col("Carries_Carries"))
fbref_matchstats = fbref_matchstats.withColumn("Carries_PrgC", col("PrgC_Carries"))
fbref_matchstats = fbref_matchstats.withColumn("Take_Ons_Att", col("Att_Take_Ons"))
fbref_matchstats = fbref_matchstats.withColumn("Take_Ons_Succ", col("Succ_Take_Ons"))

#fbref_matchstats = fbref_matchstats.withColumn("Row_Id", monotonically_increasing_id())

In [ ]:
fbref_matchstats = fbref_matchstats.select('Match_Id', 'Team', 'Is_Home_Team', 'Manager', 'Captain',
                                           match_stats['Home_Formation'], match_stats['Away_Formation'],
                                           match_stats['Home_Possession'], match_stats['Away_Possession'],
                                           match_stats['Home_Fouls'], match_stats['Away_Fouls'],
                                           match_stats['Home_Corners'], match_stats['Away_Corners'],
                                           match_stats['Home_Crosses'], match_stats['Away_Crosses'],
                                           match_stats['Home_Aerials_Won'], match_stats['Away_Aerials_Won'],
                                           match_stats['Home_Clearances'], match_stats['Away_Clearances'],
                                           match_stats['Home_Offsides'], match_stats['Away_Offsides'],
                                           match_stats['Home_Goal_Kicks'], match_stats['Away_Goal_Kicks'],
                                           match_stats['Home_Throw_Ins'], match_stats['Away_Throw_Ins'],
                                           match_stats['Home_Long_Balls'], match_stats['Away_Long_Balls'],
                                           'Total_Players_Stats', 'Minutes',
                                           'Gls', 'Ast', 'PK', 'PK_Att', 'Sh', 'SoT', 'CrdY', 'CrdR',
                                           'Touches', 'Tkl', 'Int', 'Blocks',
                                           'xG', 'npxG' ,'xAG', 'SCA', 'GCA',
                                           'Passes_Cmp', 'Passes_Att', 'Passes_Cmp_Percentage', 'Passes_PrgP',
                                           'Carries', 'Carries_PrgC', 'Take_Ons_Att', 'Take_Ons_Succ',
                                           advanced_match_stats['Home_Score'], advanced_match_stats['Away_Score'])

In [ ]:
#fbref_matchstats = fbref_matchstats.filter((fbref_matchstats.Row_Id % 2) != 0)
#fbref_matchstats = fbref_matchstats.drop("Row_Id")

In [ ]:
fbref_matchstats = fbref_matchstats.withColumn("Formation", when(col("Is_Home_Team") == "Yes", col("Home_Formation"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Formation")))
fbref_matchstats = fbref_matchstats.withColumn("Possession", when(col("Is_Home_Team") == "Yes", col("Home_Possession"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Possession")))
fbref_matchstats = fbref_matchstats.withColumn("Fouls", when(col("Is_Home_Team") == "Yes", col("Home_Fouls"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Fouls")))
fbref_matchstats = fbref_matchstats.withColumn("Corners", when(col("Is_Home_Team") == "Yes", col("Home_Corners"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Corners")))
fbref_matchstats = fbref_matchstats.withColumn("Crosses", when(col("Is_Home_Team") == "Yes", col("Home_Crosses"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Crosses")))
fbref_matchstats = fbref_matchstats.withColumn("Aerials_Won", when(col("Is_Home_Team") == "Yes", col("Home_Aerials_Won"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Aerials_Won")))
fbref_matchstats = fbref_matchstats.withColumn("Clearances", when(col("Is_Home_Team") == "Yes", col("Home_Clearances"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Clearances")))
fbref_matchstats = fbref_matchstats.withColumn("Offsides", when(col("Is_Home_Team") == "Yes", col("Home_Offsides"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Offsides")))
fbref_matchstats = fbref_matchstats.withColumn("Goal_Kicks", when(col("Is_Home_Team") == "Yes", col("Home_Goal_Kicks"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Goal_Kicks")))
fbref_matchstats = fbref_matchstats.withColumn("Throw_Ins", when(col("Is_Home_Team") == "Yes", col("Home_Throw_Ins"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Throw_Ins")))
fbref_matchstats = fbref_matchstats.withColumn("Long_Balls", when(col("Is_Home_Team") == "Yes", col("Home_Long_Balls"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Long_Balls")))
fbref_matchstats = fbref_matchstats.withColumn("Score", when(col("Is_Home_Team") == "Yes", col("Home_Score"))
                                               .when(col("Is_Home_Team") == "No", col("Away_Score")))

In [ ]:
fbref_matchstats = fbref_matchstats.select('Match_Id', 'Team', 'Is_Home_Team', 'Manager', 'Captain',
                                           'Formation', 'Possession',
                                           'Fouls', 'Corners', 'Crosses', 'Aerials_Won',
                                           'Clearances', 'Offsides', 'Goal_Kicks', 'Throw_Ins', 'Long_Balls',
                                           'Total_Players_Stats', 'Minutes',
                                           'Gls', 'Ast', 'PK', 'PK_Att', 'Sh', 'SoT', 'CrdY', 'CrdR',
                                           'Touches', 'Tkl', 'Int', 'Blocks',
                                           'xG', 'npxG' ,'xAG', 'SCA', 'GCA',
                                           'Passes_Cmp', 'Passes_Att', 'Passes_Cmp_Percentage', 'Passes_PrgP',
                                           'Carries', 'Carries_PrgC', 'Take_Ons_Att', 'Take_Ons_Succ',
                                           'Score')

In [ ]:
display(fbref_matchstats.limit(5).toPandas())

,Match_Id,Team,Is_Home_Team,Manager,Captain,Formation,Possession,Fouls,Corners,Crosses,...,GCA,Passes_Cmp,Passes_Att,Passes_Cmp_Percentage,Passes_PrgP,Carries,Carries_PrgC,Take_Ons_Att,Take_Ons_Succ,Score
0,39b738ac,Saint-Étienne,Yes,Unknown,Unknown,4-2-3-1,59,14,7,33,...,3,471,588,80.1,54,360,24,29,14,2
1,39b738ac,Strasbourg,No,Unknown,Unknown,4-1-2-1-2,41,10,4,14,...,4,293,410,71.5,23,297,18,12,9,2
2,ed0f2e13,Lille,Yes,Unknown,Unknown,4-2-3-1,50,11,3,12,...,6,408,514,79.4,41,360,24,29,17,3
3,ed0f2e13,Metz,No,Unknown,Unknown,4-2-3-1,50,12,5,14,...,2,378,511,74,38,338,15,23,12,1
4,de5b03fe,Nice,Yes,Unknown,Unknown,4/3/2003,75,14,11,34,...,0,730,826,88.4,69,527,41,40,18,0


In [ ]:
fbref_matchstats.count()

25238

<p3>FBref_MatchPlayerStats table</p3>

In [ ]:
#.join(match_stats, on="Game_URL")
fbref_matchplayerstats = match_players_stats

In [ ]:
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Match_Id", expr("substring(Game_URL, 30, 8)"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Is_Home_Team", when(col("Home_Away") == "Home", "Yes")
                                               .when(col("Home_Away") == "Away", "No").otherwise("Unknown"))

fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Player_Name", col("Player"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Player_Kitnum", col("Player_Num"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Nationality", col("Nation"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Position", col("Pos"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Age", col("Age"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Minutes", col("Min"))

fbref_matchplayerstats = fbref_matchplayerstats.withColumn("PK_Att", col("PKatt"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("xG", col("xG_Expected"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("npxG", col("npxG_Expected"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("xAG", col("xAG_Expected"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("SCA", col("SCA_SCA"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("GCA", col("GCA_SCA"))

fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Passes_Cmp", col("Cmp_Passes"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Passes_Att", col("Att_Passes"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Passes_Cmp_Percentage", col("Cmp_percent_Passes"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Passes_PrgP", col("PrgP_Passes"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Carries", col("Carries_Carries"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Carries_PrgC", col("PrgC_Carries"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Take_Ons_Att", col("Att_Take_Ons"))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn("Take_Ons_Succ", col("Succ_Take_Ons"))

#fbref_matchstats = fbref_matchstats.withColumn("Row_Id", monotonically_increasing_id())

In [ ]:
fbref_matchplayerstats = fbref_matchplayerstats.select('Match_Id', 'Team', 'Is_Home_Team', 'Player_Name', 'Player_Kitnum',
                                                       'Nationality', 'Position', 'Age', 'Minutes',
                                                       'Gls', 'Ast', 'PK', 'PK_Att', 'Sh', 'SoT', 'CrdY', 'CrdR',
                                                       'Touches', 'Tkl', 'Int', 'Blocks',
                                                       'xG', 'npxG' ,'xAG', 'SCA', 'GCA',
                                                       'Passes_Cmp', 'Passes_Att', 'Passes_Cmp_Percentage', 'Passes_PrgP',
                                                       'Carries', 'Carries_PrgC', 'Take_Ons_Att', 'Take_Ons_Succ')

In [ ]:
display(fbref_matchplayerstats.limit(5).toPandas())

,Match_Id,Team,Is_Home_Team,Player_Name,Player_Kitnum,Nationality,Position,Age,Minutes,Gls,...,SCA,GCA,Passes_Cmp,Passes_Att,Passes_Cmp_Percentage,Passes_PrgP,Carries,Carries_PrgC,Take_Ons_Att,Take_Ons_Succ
0,a68e623d,Monaco,Yes,Kylian Mbappé,10,FRA,"FW,LM",18-227,74,0,...,3,1,19,29,65.5,3,27,6,6,4
1,a68e623d,Monaco,Yes,Allan Saint-Maximin,12,FRA,RM,20-145,16,0,...,2,0,3,4,75,0,5,1,0,0
2,a68e623d,Monaco,Yes,Radamel Falcao,9,COL,FW,31-175,86,1,...,0,0,20,25,80,1,24,1,2,1
3,a68e623d,Monaco,Yes,Youri Tielemans,17,BEL,FW,20-089,4,0,...,1,0,4,4,100,1,3,0,0,0
4,a68e623d,Monaco,Yes,Thomas Lemar,27,FRA,"LM,RM",21-265,90,0,...,4,0,42,56,75,6,44,2,5,1


In [ ]:
fbref_matchplayerstats.count()

369519

Tên các matches

In [ ]:
fbref_matchinfos_teams = fbref_matchinfos.select('Home_Team').distinct()
fbref_matchsquad_teams = fbref_matchsquad.select('Team').distinct()
fbref_matchgoals_teams = fbref_matchgoals.select('Team').distinct()
fbref_matchstats_teams = fbref_matchstats.select('Team').distinct()
fbref_matchplayerstats_teams = fbref_matchplayerstats.select('Team').distinct()

In [ ]:
different_teams = fbref_matchinfos_teams.exceptAll(fbref_matchsquad_teams).collect()
for value in different_teams:
    print(value[0])

Tottenham
Brighton
M'Gladbach
Betis
Gladbach
Sheffield Utd
Paris S-G
Newcastle Utd
Leverkusen
La Coruña
Manchester Utd
West Ham
Inter
Nott'ham Forest
Wolves
West Brom
Eint Frankfurt
Huddersfield


In [ ]:
different_teams = fbref_matchinfos_teams.exceptAll(fbref_matchgoals_teams).collect()
for value in different_teams:
    print(value[0])

Tottenham
Brighton
M'Gladbach
Betis
Gladbach
Sheffield Utd
Paris S-G
Newcastle Utd
Leverkusen
La Coruña
Manchester Utd
West Ham
Inter
Nott'ham Forest
Wolves
West Brom
Eint Frankfurt
Huddersfield


In [ ]:
different_teams = fbref_matchinfos_teams.exceptAll(fbref_matchstats_teams).collect()
for value in different_teams:
    print(value[0])

Tottenham
Brighton
M'Gladbach
Betis
Gladbach
Sheffield Utd
Paris S-G
Newcastle Utd
Leverkusen
La Coruña
Manchester Utd
West Ham
Inter
Nott'ham Forest
Wolves
West Brom
Eint Frankfurt
Huddersfield


In [ ]:
different_teams = fbref_matchinfos_teams.exceptAll(fbref_matchplayerstats_teams).collect()
for value in different_teams:
    print(value[0])

Tottenham
Brighton
M'Gladbach
Betis
Gladbach
Sheffield Utd
Paris S-G
Newcastle Utd
Leverkusen
La Coruña
Manchester Utd
West Ham
Inter
Nott'ham Forest
Wolves
West Brom
Eint Frankfurt
Huddersfield


In [ ]:
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Newcastle United', 'Newcastle Utd')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Tottenham Hotspur', 'Tottenham')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Internazionale', 'Inter')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'West Bromwich Albion', 'West Brom')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Nottingham Forest', "Nott'ham Forest")
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Brighton & Hove Albion', 'Brighton')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Real Betis', 'Betis')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Paris Saint-Germain', 'Paris S-G')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Bayer Leverkusen', 'Leverkusen')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Wolverhampton Wanderers', 'Wolves')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Mönchengladbach', "M'Gladbach")
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Sheffield United', 'Sheffield Utd')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Deportivo La Coruña', 'La Coruña')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Manchester United', 'Manchester Utd')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'West Ham United', 'West Ham')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Eintracht Frankfurt', 'Eint Frankfurt')
                                               .otherwise(col('Team')))
fbref_matchsquad = fbref_matchsquad.withColumn('Team', when(col('Team') == 'Huddersfield Town', 'Huddersfield')
                                               .otherwise(col('Team')))

In [ ]:
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Newcastle United', 'Newcastle Utd')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Tottenham Hotspur', 'Tottenham')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Internazionale', 'Inter')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'West Bromwich Albion', 'West Brom')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Nottingham Forest', "Nott'ham Forest")
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Brighton & Hove Albion', 'Brighton')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Real Betis', 'Betis')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Paris Saint-Germain', 'Paris S-G')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Bayer Leverkusen', 'Leverkusen')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Wolverhampton Wanderers', 'Wolves')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Mönchengladbach', "M'Gladbach")
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Sheffield United', 'Sheffield Utd')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Deportivo La Coruña', 'La Coruña')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Manchester United', 'Manchester Utd')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'West Ham United', 'West Ham')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Eintracht Frankfurt', 'Eint Frankfurt')
                                               .otherwise(col('Team')))
fbref_matchgoals = fbref_matchgoals.withColumn('Team', when(col('Team') == 'Huddersfield Town', 'Huddersfield')
                                               .otherwise(col('Team')))

In [ ]:
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Newcastle United', 'Newcastle Utd')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Tottenham Hotspur', 'Tottenham')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Internazionale', 'Inter')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'West Bromwich Albion', 'West Brom')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Nottingham Forest', "Nott'ham Forest")
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Brighton & Hove Albion', 'Brighton')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Real Betis', 'Betis')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Paris Saint-Germain', 'Paris S-G')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Bayer Leverkusen', 'Leverkusen')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Wolverhampton Wanderers', 'Wolves')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Mönchengladbach', "M'Gladbach")
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Sheffield United', 'Sheffield Utd')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Deportivo La Coruña', 'La Coruña')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Manchester United', 'Manchester Utd')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'West Ham United', 'West Ham')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Eintracht Frankfurt', 'Eint Frankfurt')
                                               .otherwise(col('Team')))
fbref_matchstats = fbref_matchstats.withColumn('Team', when(col('Team') == 'Huddersfield Town', 'Huddersfield')
                                               .otherwise(col('Team')))

In [ ]:
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Newcastle United', 'Newcastle Utd')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Tottenham Hotspur', 'Tottenham')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Internazionale', 'Inter')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'West Bromwich Albion', 'West Brom')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Nottingham Forest', "Nott'ham Forest")
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Brighton & Hove Albion', 'Brighton')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Real Betis', 'Betis')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Paris Saint-Germain', 'Paris S-G')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Bayer Leverkusen', 'Leverkusen')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Wolverhampton Wanderers', 'Wolves')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Mönchengladbach', "M'Gladbach")
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Sheffield United', 'Sheffield Utd')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Deportivo La Coruña', 'La Coruña')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Manchester United', 'Manchester Utd')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'West Ham United', 'West Ham')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Eintracht Frankfurt', 'Eint Frankfurt')
                                               .otherwise(col('Team')))
fbref_matchplayerstats = fbref_matchplayerstats.withColumn('Team', when(col('Team') == 'Huddersfield Town', 'Huddersfield')
                                               .otherwise(col('Team')))

In [ ]:
#fbref_matchplayerstats.coalesce(1).write.option("header", "true").csv("/content/drive/MyDrive/Colab Notebooks/fbref/test.csv")

In [ ]:
different_teams = fbref_matchinfos_teams.exceptAll(fbref_matchstats_teams).collect()
for value in different_teams:
    print(value[0])

Tottenham
Brighton
M'Gladbach
Betis
Gladbach
Sheffield Utd
Paris S-G
Newcastle Utd
Leverkusen
La Coruña
Manchester Utd
West Ham
Inter
Nott'ham Forest
Wolves
West Brom
Eint Frankfurt
Huddersfield


<h3>2. Sofifa tables</h3>

In [ ]:
sofifa_players_attr_all = spark.read.csv("gs://football_data_etl/football_data_extracted/sofifa/player_attr_all.csv",
                                 header=True, inferSchema=True)
sofifa_players_attr_eng = spark.read.csv("gs://football_data_etl/football_data_extracted/sofifa/player_attr_eng.csv",
                                 header=True, inferSchema=True)
#sofifa_players_attr_more = spark.read.csv("gs://football-data-etl/football-data-extracted/sofifa/player_attr_more.csv",
#                                 header=True, inferSchema=True)

<p3>Sofifa_PlayerAttr table</p3>

In [ ]:
display(sofifa_players_attr_all.limit(5).toPandas())

,acceleration,age,aggression,agility,all_positions,attacking_work_rate,balance,ball_control,birthday,body_type,...,stamina,standing_tackle,strength,update_date,value,vision,volleys,wage,weak_foot,weight
0,79,27,77,79,RB RWB,High,77,75,1993-04-26,Normal (170-185),...,81,77,65,"Sep 23, 2020",€15M,65,52,€46K,3,71
1,80,27,48,75,ST LM,High,71,79,1992-09-14,Normal (185+),...,72,14,76,"Sep 23, 2020",€15M,65,73,€65K,4,78
2,89,24,62,81,LM,High,81,81,1996-03-26,Normal (170-185),...,74,37,60,"Sep 23, 2020",€21M,74,68,€33K,4,70
3,32,36,83,47,CB,Medium,53,64,1983-12-22,Lean (185+),...,59,79,82,"Sep 23, 2020",€2.5M,55,34,€24K,3,81
4,30,24,25,32,GK,Medium,29,18,1995-10-31,Normal (185+),...,26,10,66,"Sep 23, 2020",€21.5M,42,8,€20K,2,85


In [ ]:
for row in sofifa_players_attr_all.head(5):
    print(row)

Row(acceleration='79', age='27', aggression='77', agility=79, all_positions='RB RWB', attacking_work_rate='High', balance=77, ball_control=75, birthday=datetime.date(1993, 4, 26), body_type=' Normal (170-185)', club='Monaco', club_contract='2024', club_joined='Aug 6, 2019', club_kitnum=26, club_league='Ligue 1', club_loaned_from=None, club_position='RB', club_rating=7, composure=70, crossing=75, curve=62, defensive_awareness=None, defensive_work_rate='Medium', dribbling=76, finishing=49, fk_accuracy=33, gk_diving=15, gk_handling=8, gk_kicking=16, gk_positioning=10, gk_reflexes=13, heading_accuracy=60, height=172, interceptions=75, jumping=80, long_passing=70, long_shots=45, marking=None, national_team=None, national_team_kitnum=None, national_team_position=None, national_team_rating=None, nationality='France', overall_rating=78, penalties=38, play_styles=None, player_full_name='Ruben Aguilar', player_name='R. Aguilar', positioning=65, potential=79, preferred_foot='Right', reactions=73,

In [ ]:
for row in sofifa_players_attr_all.tail(5):
    print(row)

Row(acceleration='88', age='26', aggression='64', agility=72, all_positions='LM CAM RM', attacking_work_rate='Medium', balance=68, ball_control=72, birthday=datetime.date(1996, 12, 20), body_type=' Lean (185+)', club='Darmstadt 98', club_contract='2024', club_joined='Jul 5, 2019', club_kitnum=18, club_league='Bundesliga', club_loaned_from=None, club_position='RES', club_rating=7, composure=55, crossing=65, curve=55, defensive_awareness=None, defensive_work_rate='Medium', dribbling=71, finishing=64, fk_accuracy=47, gk_diving=5, gk_handling=9, gk_kicking=7, gk_positioning=11, gk_reflexes=12, heading_accuracy=49, height=188, interceptions=54, jumping=75, long_passing=57, long_shots=61, marking=None, national_team=None, national_team_kitnum=None, national_team_position=None, national_team_rating=None, nationality='Austria', overall_rating=70, penalties=44, play_styles='#Rapid#Quick Step', player_full_name='Mathias Honsak', player_name='M. Honsak', positioning=70, potential=70, preferred_fo

In [ ]:
#sofifa_players_attr = sofifa_players_attr.union(sofifa_players_attr_more)

In [ ]:
sofifa_players_attr_all.count()

234509

In [ ]:
new_column_names = [col.replace('_', ' ') for col in sofifa_players_attr_all.columns]
sofifa_players_attr_all = sofifa_players_attr_all.toDF(*new_column_names)

new_columns = [col(column).alias(' '.join(word.capitalize() for word in column.split(' '))) for column in sofifa_players_attr_all.columns]
sofifa_players_attr_all = sofifa_players_attr_all.select(*new_columns)

new_column_names = [col.replace(' ', '_') for col in sofifa_players_attr_all.columns]
sofifa_players_attr_all = sofifa_players_attr_all.toDF(*new_column_names)

sofifa_players_attr_all = sofifa_players_attr_all.withColumn('Birthday', to_timestamp(col('Birthday'), 'yyyy-MM-dd'))
#sofifa_players_attr = sofifa_players_attr.withColumn('Club_Joined', to_timestamp(col('Club_Joined'), 'yyyy-MM-dd'))
#sofifa_players_attr = sofifa_players_attr.withColumn('Update_Date', to_timestamp(col('Update_Date'), 'yyyy-MM-dd'))

In [ ]:
#update_date
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Year", element_at(split(sofifa_players_attr_all.Update_Date, " "), -1))
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Month", split(sofifa_players_attr_all.Update_Date, " ").getItem(0))
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Day", split(sofifa_players_attr_all.Update_Date, " ").getItem(1))
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Day", regexp_replace(sofifa_players_attr_all.Day, ",", ""))

sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Day", when(col("Day") == "1", lit("01"))
                                                     .when(col("Day") == "2", lit("02"))
                                                     .when(col("Day") == "3", lit("03"))
                                                     .when(col("Day") == "4", lit("04"))
                                                     .when(col("Day") == "5", lit("05"))
                                                     .when(col("Day") == "6", lit("06"))
                                                     .when(col("Day") == "7", lit("07"))
                                                     .when(col("Day") == "8", lit("08"))
                                                     .when(col("Day") == "9", lit("09"))
                                                     .otherwise(col("Day")))
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Month", when(col("Month") == "Jan", lit("01"))
                                                     .when(col("Month") == "Feb", lit("02"))
                                                     .when(col("Month") == "Mar", lit("03"))
                                                     .when(col("Month") == "Apr", lit("04"))
                                                     .when(col("Month") == "May", lit("05"))
                                                     .when(col("Month") == "Jun", lit("06"))
                                                     .when(col("Month") == "Jul", lit("07"))
                                                     .when(col("Month") == "Aug", lit("08"))
                                                     .when(col("Month") == "Sep", lit("09"))
                                                     .when(col("Month") == "Oct", lit("10"))
                                                     .when(col("Month") == "Nov", lit("11"))
                                                     .when(col("Month") == "Dec", lit("12")))

sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Update_Date", concat(sofifa_players_attr_all.Year, lit("-"),
                                                                           sofifa_players_attr_all.Month, lit("-"),
                                                                           sofifa_players_attr_all.Day))
columns_to_remove = ["Year", "Month", "Day"]
sofifa_players_attr_all = sofifa_players_attr_all.drop(*columns_to_remove)

In [ ]:
sofifa_players_attr_all = sofifa_players_attr_all.withColumn('Update_Date', to_timestamp(col('Update_Date'), 'yyyy-MM-dd'))

In [ ]:
#club_joined
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Year", element_at(split(sofifa_players_attr_all.Club_Joined, " "), -1))
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Month", split(sofifa_players_attr_all.Club_Joined, " ").getItem(0))
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Day", split(sofifa_players_attr_all.Club_Joined, " ").getItem(1))
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Day", regexp_replace(sofifa_players_attr_all.Day, ",", ""))

sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Day", when(col("Day") == "1", lit("01"))
                                                     .when(col("Day") == "2", lit("02"))
                                                     .when(col("Day") == "3", lit("03"))
                                                     .when(col("Day") == "4", lit("04"))
                                                     .when(col("Day") == "5", lit("05"))
                                                     .when(col("Day") == "6", lit("06"))
                                                     .when(col("Day") == "7", lit("07"))
                                                     .when(col("Day") == "8", lit("08"))
                                                     .when(col("Day") == "9", lit("09"))
                                                     .otherwise(col("Day")))
sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Month", when(col("Month") == "Jan", lit("01"))
                                                     .when(col("Month") == "Feb", lit("02"))
                                                     .when(col("Month") == "Mar", lit("03"))
                                                     .when(col("Month") == "Apr", lit("04"))
                                                     .when(col("Month") == "May", lit("05"))
                                                     .when(col("Month") == "Jun", lit("06"))
                                                     .when(col("Month") == "Jul", lit("07"))
                                                     .when(col("Month") == "Aug", lit("08"))
                                                     .when(col("Month") == "Sep", lit("09"))
                                                     .when(col("Month") == "Oct", lit("10"))
                                                     .when(col("Month") == "Nov", lit("11"))
                                                     .when(col("Month") == "Dec", lit("12")))

sofifa_players_attr_all = sofifa_players_attr_all.withColumn("Club_Joined", concat(sofifa_players_attr_all.Year, lit("-"),
                                                                           sofifa_players_attr_all.Month, lit("-"),
                                                                           sofifa_players_attr_all.Day))
columns_to_remove = ["Year", "Month", "Day"]
sofifa_players_attr_all = sofifa_players_attr_all.drop(*columns_to_remove)

In [ ]:
sofifa_players_attr_all = sofifa_players_attr_all.withColumn('Club_Joined', to_timestamp(col('Club_Joined'), 'yyyy-MM-dd'))

In [ ]:
for row in sofifa_players_attr_all.head(5):
    print(row)

Row(Acceleration='79', Age='27', Aggression='77', Agility=79, All_Positions='RB RWB', Attacking_Work_Rate='High', Balance=77, Ball_Control=75, Birthday=datetime.datetime(1993, 4, 26, 0, 0), Body_Type=' Normal (170-185)', Club='Monaco', Club_Contract='2024', Club_Joined=datetime.datetime(2019, 8, 6, 0, 0), Club_Kitnum=26, Club_League='Ligue 1', Club_Loaned_From=None, Club_Position='RB', Club_Rating=7, Composure=70, Crossing=75, Curve=62, Defensive_Awareness=None, Defensive_Work_Rate='Medium', Dribbling=76, Finishing=49, Fk_Accuracy=33, Gk_Diving=15, Gk_Handling=8, Gk_Kicking=16, Gk_Positioning=10, Gk_Reflexes=13, Heading_Accuracy=60, Height=172, Interceptions=75, Jumping=80, Long_Passing=70, Long_Shots=45, Marking=None, National_Team=None, National_Team_Kitnum=None, National_Team_Position=None, National_Team_Rating=None, Nationality='France', Overall_Rating=78, Penalties=38, Play_Styles=None, Player_Full_Name='Ruben Aguilar', Player_Name='R. Aguilar', Positioning=65, Potential=79, Prefe

In [ ]:
sofifa_players_attr_all.printSchema()

root
 |-- Acceleration: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Aggression: string (nullable = true)
 |-- Agility: integer (nullable = true)
 |-- All_Positions: string (nullable = true)
 |-- Attacking_Work_Rate: string (nullable = true)
 |-- Balance: integer (nullable = true)
 |-- Ball_Control: integer (nullable = true)
 |-- Birthday: timestamp (nullable = true)
 |-- Body_Type: string (nullable = true)
 |-- Club: string (nullable = true)
 |-- Club_Contract: string (nullable = true)
 |-- Club_Joined: timestamp (nullable = true)
 |-- Club_Kitnum: integer (nullable = true)
 |-- Club_League: string (nullable = true)
 |-- Club_Loaned_From: string (nullable = true)
 |-- Club_Position: string (nullable = true)
 |-- Club_Rating: integer (nullable = true)
 |-- Composure: integer (nullable = true)
 |-- Crossing: integer (nullable = true)
 |-- Curve: integer (nullable = true)
 |-- Defensive_Awareness: string (nullable = true)
 |-- Defensive_Work_Rate: string (nullable = t

In [ ]:
display(sofifa_players_attr_all.limit(50).toPandas())

,Acceleration,Age,Aggression,Agility,All_Positions,Attacking_Work_Rate,Balance,Ball_Control,Birthday,Body_Type,...,Stamina,Standing_Tackle,Strength,Update_Date,Value,Vision,Volleys,Wage,Weak_Foot,Weight
0,79,27,77,79,RB RWB,High,77,75,1993-04-26,Normal (170-185),...,81,77,65,2020-09-23,€15M,65,52,€46K,3,71
1,80,27,48,75,ST LM,High,71,79,1992-09-14,Normal (185+),...,72,14,76,2020-09-23,€15M,65,73,€65K,4,78
2,89,24,62,81,LM,High,81,81,1996-03-26,Normal (170-185),...,74,37,60,2020-09-23,€21M,74,68,€33K,4,70
3,32,36,83,47,CB,Medium,53,64,1983-12-22,Lean (185+),...,59,79,82,2020-09-23,€2.5M,55,34,€24K,3,81
4,30,24,25,32,GK,Medium,29,18,1995-10-31,Normal (185+),...,26,10,66,2020-09-23,€21.5M,42,8,€20K,2,85
5,70,20,78,70,CDM CB,Low,68,78,1999-11-23,Lean (170-185),...,77,79,71,2020-09-23,€36.5M,73,28,€29K,3,68
6,67,30,89,65,CB,Medium,71,64,1990-01-08,Lean (170-185),...,76,80,75,2020-09-23,€14.5M,39,31,€40K,3,75
7,79,28,79,67,RWB RB LB,High,70,77,1991-10-03,Normal (170-185),...,83,73,71,2020-09-23,€16M,71,44,€34K,3,78
8,37,28,29,36,GK,Medium,34,33,1992-02-26,Normal (185+),...,30,15,67,2020-09-23,€15M,49,19,€28K,2,88
9,36,28,22,48,GK,Medium,27,11,1992-03-01,Lean (185+),...,35,11,72,2020-09-23,€15M,39,9,€40K,2,86


In [ ]:
display(sofifa_players_attr_eng.limit(5).toPandas())

,acceleration,age,aggression,agility,all_positions,attacking_work_rate,balance,ball_control,birthday,body_type,...,stamina,standing_tackle,strength,update_date,value,vision,volleys,wage,weak_foot,weight
0,56,19,55,41,CB,Low,47,26,2004-01-08,Lean (170-185),...,63,52,63,2023-09-22,€100K,34,26,€2K,3,80
1,57,17,45,44,CB,Medium,64,30,2000-01-06,Lean (170-185),...,62,52,45,2017-09-18,€60K,27,20,€6K,3,67
2,68,19,50,56,RB,Medium,66,54,1998-12-18,Normal (170-185),...,53,55,56,2018-08-21,€110K,33,27,€3K,3,70
3,63,17,33,59,ST,Medium,49,44,2001-07-22,Normal (185+),...,50,17,62,2019-09-19,€70K,33,50,€1K,2,77
4,65,19,57,56,CM,Medium,68,53,1999-09-02,Lean (170-185),...,53,41,48,2019-09-19,€110K,49,40,€4K,3,70


In [ ]:
for row in sofifa_players_attr_eng.head(5):
    print(row)

Row(acceleration=56, age=19, aggression=55, agility=41, all_positions='CB', attacking_work_rate='Low', balance=47, ball_control=26, birthday=datetime.date(2004, 1, 8), body_type='Lean (170-185)', club='Crystal Palace', club_contract='2027', club_joined='Aug 3, 2022', club_kitnum=42, club_league='Premier League', club_loaned_from=None, club_position='RES', club_rating=76, composure=40.0, crossing=25, curve=27, defensive_awareness=54.0, defensive_work_rate='Medium', dribbling=24, finishing=19, fk_accuracy=22, gk_diving=8, gk_handling=8, gk_kicking=7, gk_positioning=11, gk_reflexes=10, heading_accuracy=42, height=184, interceptions=50, jumping=49, long_passing=38, long_shots=16, marking=None, national_team=None, national_team_kitnum=None, national_team_position=None, national_team_rating=None, nationality='Republic of Ireland', overall_rating=50, penalties=36, play_styles=None, player_full_name='Seán Grehan', player_name='S. Grehan', positioning=29, potential=65, preferred_foot='Right', r

In [ ]:
new_column_names = [col.replace('_', ' ') for col in sofifa_players_attr_eng.columns]
sofifa_players_attr_eng = sofifa_players_attr_eng.toDF(*new_column_names)

new_columns = [col(column).alias(' '.join(word.capitalize() for word in column.split(' '))) for column in sofifa_players_attr_eng.columns]
sofifa_players_attr_eng = sofifa_players_attr_eng.select(*new_columns)

new_column_names = [col.replace(' ', '_') for col in sofifa_players_attr_eng.columns]
sofifa_players_attr_eng = sofifa_players_attr_eng.toDF(*new_column_names)

sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn('Birthday', to_timestamp(col('Birthday'), 'yyyy-MM-dd'))
#sofifa_players_attr = sofifa_players_attr.withColumn('Club_Joined', to_timestamp(col('Club_Joined'), 'yyyy-MM-dd'))
sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn('Update_Date', to_timestamp(col('Update_Date'), 'yyyy-MM-dd'))

In [ ]:
#update_date
sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn("Year", element_at(split(sofifa_players_attr_eng.Club_Joined, " "), -1))
sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn("Month", split(sofifa_players_attr_eng.Club_Joined, " ").getItem(0))
sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn("Day", split(sofifa_players_attr_eng.Club_Joined, " ").getItem(1))
sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn("Day", regexp_replace(sofifa_players_attr_eng.Day, ",", ""))

sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn("Day", when(col("Day") == "1", lit("01"))
                                                     .when(col("Day") == "2", lit("02"))
                                                     .when(col("Day") == "3", lit("03"))
                                                     .when(col("Day") == "4", lit("04"))
                                                     .when(col("Day") == "5", lit("05"))
                                                     .when(col("Day") == "6", lit("06"))
                                                     .when(col("Day") == "7", lit("07"))
                                                     .when(col("Day") == "8", lit("08"))
                                                     .when(col("Day") == "9", lit("09"))
                                                     .otherwise(col("Day")))
sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn("Month", when(col("Month") == "Jan", lit("01"))
                                                     .when(col("Month") == "Feb", lit("02"))
                                                     .when(col("Month") == "Mar", lit("03"))
                                                     .when(col("Month") == "Apr", lit("04"))
                                                     .when(col("Month") == "May", lit("05"))
                                                     .when(col("Month") == "Jun", lit("06"))
                                                     .when(col("Month") == "Jul", lit("07"))
                                                     .when(col("Month") == "Aug", lit("08"))
                                                     .when(col("Month") == "Sep", lit("09"))
                                                     .when(col("Month") == "Oct", lit("10"))
                                                     .when(col("Month") == "Nov", lit("11"))
                                                     .when(col("Month") == "Dec", lit("12")))

sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn("Club_Joined", concat(sofifa_players_attr_eng.Year, lit("-"),
                                                                           sofifa_players_attr_eng.Month, lit("-"),
                                                                           sofifa_players_attr_eng.Day))
columns_to_remove = ["Year", "Month", "Day"]
sofifa_players_attr_eng = sofifa_players_attr_eng.drop(*columns_to_remove)

In [ ]:
sofifa_players_attr_eng = sofifa_players_attr_eng.withColumn('Club_Joined', to_timestamp(col('Club_Joined'), 'yyyy-MM-dd'))

In [ ]:
for row in sofifa_players_attr_eng.head(5):
    print(row)

Row(Acceleration=56, Age=19, Aggression=55, Agility=41, All_Positions='CB', Attacking_Work_Rate='Low', Balance=47, Ball_Control=26, Birthday=datetime.datetime(2004, 1, 8, 0, 0), Body_Type='Lean (170-185)', Club='Crystal Palace', Club_Contract='2027', Club_Joined=datetime.datetime(2022, 8, 3, 0, 0), Club_Kitnum=42, Club_League='Premier League', Club_Loaned_From=None, Club_Position='RES', Club_Rating=76, Composure=40.0, Crossing=25, Curve=27, Defensive_Awareness=54.0, Defensive_Work_Rate='Medium', Dribbling=24, Finishing=19, Fk_Accuracy=22, Gk_Diving=8, Gk_Handling=8, Gk_Kicking=7, Gk_Positioning=11, Gk_Reflexes=10, Heading_Accuracy=42, Height=184, Interceptions=50, Jumping=49, Long_Passing=38, Long_Shots=16, Marking=None, National_Team=None, National_Team_Kitnum=None, National_Team_Position=None, National_Team_Rating=None, Nationality='Republic of Ireland', Overall_Rating=50, Penalties=36, Play_Styles=None, Player_Full_Name='Seán Grehan', Player_Name='S. Grehan', Positioning=29, Potenti

In [ ]:
#union
sofifa_players_attr = sofifa_players_attr_all.union(sofifa_players_attr_eng)

In [ ]:
sofifa_players_attr.count()

291641

Đổi tên Team

In [ ]:
teams_fbref = fbref_matchsquad.select('Team').distinct()
teams_sofifa = sofifa_players_attr.select('Club').distinct()

In [ ]:
teams_fbref.collect()

[Row(Team='Nice'),
 Row(Team='Montpellier'),
 Row(Team='Dijon'),
 Row(Team='Angers'),
 Row(Team='Lille'),
 Row(Team='Nantes'),
 Row(Team='Paris S-G'),
 Row(Team='Marseille'),
 Row(Team='Caen'),
 Row(Team='Clermont Foot'),
 Row(Team='Lens'),
 Row(Team='Lyon'),
 Row(Team='Saint-Étienne'),
 Row(Team='Monaco'),
 Row(Team='Bordeaux'),
 Row(Team='Nîmes'),
 Row(Team='Troyes'),
 Row(Team='Strasbourg'),
 Row(Team='Toulouse'),
 Row(Team='Guingamp'),
 Row(Team='Reims'),
 Row(Team='Rennes'),
 Row(Team='Metz'),
 Row(Team='Brest'),
 Row(Team='Lorient'),
 Row(Team='Amiens'),
 Row(Team='Le Havre'),
 Row(Team='Auxerre'),
 Row(Team='Ajaccio'),
 Row(Team='Tottenham'),
 Row(Team='Brighton'),
 Row(Team='Sheffield Utd'),
 Row(Team='Cardiff City'),
 Row(Team='Arsenal'),
 Row(Team='Brentford'),
 Row(Team='Newcastle Utd'),
 Row(Team='Leeds United'),
 Row(Team='Crystal Palace'),
 Row(Team='Burnley'),
 Row(Team='Aston Villa'),
 Row(Team='Manchester City'),
 Row(Team='Bournemouth'),
 Row(Team='Manchester Utd'),
 

In [ ]:
teams_sofifa.collect()

[Row(Club='Espanyol'),
 Row(Club='Málaga'),
 Row(Club='SC Freiburg'),
 Row(Club='Hellas Verona'),
 Row(Club='Nice'),
 Row(Club='Montpellier'),
 Row(Club='Dijon'),
 Row(Club='Manchester United'),
 Row(Club='FC Augsburg'),
 Row(Club='Bologna'),
 Row(Club='Arsenal'),
 Row(Club='Brentford'),
 Row(Club='Nantes'),
 Row(Club='Leganés'),
 Row(Club='Sheffield United'),
 Row(Club='Le Havre'),
 Row(Club='Lazio'),
 Row(Club='Real Valladolid'),
 Row(Club='RB Leipzig'),
 Row(Club='Olympique Lyonnais'),
 Row(Club='Werder Bremen'),
 Row(Club='Newcastle United'),
 Row(Club='Villarreal'),
 Row(Club='Crystal Palace'),
 Row(Club='Real Madrid'),
 Row(Club='AFC Bournemouth'),
 Row(Club='Burnley'),
 Row(Club='Athletic Club'),
 Row(Club='FC Köln'),
 Row(Club='Aston Villa'),
 Row(Club='Caen'),
 Row(Club='Paris Saint Germain'),
 Row(Club='Manchester City'),
 Row(Club='Lens'),
 Row(Club='Monza'),
 Row(Club='Elche'),
 Row(Club='Sassuolo'),
 Row(Club='Genoa'),
 Row(Club='Roma'),
 Row(Club='Lecce'),
 Row(Club='Mila

In [ ]:
different_teams = teams_fbref.exceptAll(teams_sofifa).collect()
for value in different_teams:
    print(value[0])

Tottenham
Brighton
M'Gladbach
Union Berlin
Arminia
Betis
Sheffield Utd
Angers
Lille
Celta Vigo
Paris S-G
Bochum
Marseille
Hoffenheim
Newcastle Utd
Düsseldorf
Leverkusen
Clermont Foot
La Coruña
Freiburg
Bournemouth
Lyon
Greuther Fürth
Paderborn 07
Manchester Utd
Barcelona
Mainz 05
West Ham
Nott'ham Forest
Wolfsburg
Hamburger SV
Wolves
West Brom
Eint Frankfurt
Eibar
Stuttgart
Reims
Huddersfield
Bayern Munich
Brest
Augsburg
Köln
Alavés
Dortmund
Amiens
Valladolid


Đổi tên các tên Team

In [ ]:
#Premier League
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'AFC Bournemouth',
                                                          'Bournemouth').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Newcastle United',
                                                          'Newcastle Utd').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Tottenham Hotspur',
                                                          'Tottenham').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'West Bromwich Albion',
                                                          'West Brom').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Nottingham Forest',
                                                          "Nott'ham Forest").otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Brighton & Hove Albion',
                                                          "Brighton").otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Wolverhampton Wanderers',
                                                          "Wolves").otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Sheffield United',
                                                          "Sheffield Utd").otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Manchester United',
                                                          "Manchester Utd").otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'West Ham United',
                                                          "West Ham").otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Huddersfield Town',
                                                          "Huddersfield").otherwise(col('Club')))
#Seria A
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Inter',
                                                          'Inter').otherwise(col('Club')))
#Ligue 1
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Angers SCO',
                                                          'Angers').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Paris Saint Germain',
                                                          'Paris S-G').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Clermont',
                                                          'Clermont Foot').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'LOSC Lille',
                                                          'Lille').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Olympique de Marseille',
                                                          'Marseille').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Olympique Lyonnais',
                                                          'Lyon').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Stade de Reims ',
                                                          'Reims').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Stade Brestois 29',
                                                          'Brest').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Amiens SC',
                                                          'Amiens').otherwise(col('Club')))
#La Liga
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Celta de Vigo',
                                                          'Celta Vigo').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'SD Eibar',
                                                          'Eibar').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Deportivo Alavés',
                                                          'Alavés').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Real Valladolid',
                                                          'Valladolid').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'FC Barcelona',
                                                          'Barcelona').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Real Betis',
                                                          'Betis').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Deportivo La Coruña',
                                                          'La Coruña').otherwise(col('Club')))
#Bundesliga
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'SC Freiburg',
                                                          'Freiburg').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'VfB Stuttgart',
                                                          'Stuttgart').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Bayer 04 Leverkusen',
                                                          'Leverkusen').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Borussia Dortmund',
                                                          'Dortmund').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'VfL Bochum 1848',
                                                          'Bochum').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'TSG Hoffenheim',
                                                          'Hoffenheim').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'VfL Wolfsburg',
                                                          'Wolfsburg').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'FC Köln',
                                                          'Köln').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'FC Union Berlin',
                                                          'Union Berlin').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'FSV Mainz 05',
                                                          'Mainz 05').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'FC Augsburg',
                                                          'Augsburg').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'FC Bayern München',
                                                          'Bayern Munich').otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Borussia Mönchengladbach',
                                                          "M'Gladbach").otherwise(col('Club')))
sofifa_players_attr = sofifa_players_attr.withColumn('Club',
                                                     when(col('Club') == 'Eintracht Frankfurt',
                                                          'Eint Frankfurt').otherwise(col('Club')))

In [ ]:
different_teams = teams_fbref.exceptAll(sofifa_players_attr.select('Club').distinct()).collect()
for value in different_teams:
    print(value[0])

Arminia
Düsseldorf
Greuther Fürth
Paderborn 07
Hamburger SV


<h2>II. Model Dataset</h2>

Existing files

In [ ]:
#sofifa_players_attr.coalesce(1).write.option("header", "true").csv("gs://football-data-etl/football-data-extracted/output/sofifa_players_attr_full.csv")

In [ ]:
#fbref_matchsquad = spark.read.csv("gs://football-data-etl/football-data-extracted/output/fbref_matchsquad.csv/part-00000-2dc2a400-1e54-481a-8bc2-d62e2fd490ea-c000.csv",
#                                 header=True, inferSchema=True)
#fbref_matchinfos = spark.read.csv("gs://football-data-etl/football-data-extracted/output/fbref_matchinfos.csv/part-00000-2715a1a1-c80e-403d-b63d-7ba9576dd5df-c000.csv",
#                                 header=True, inferSchema=True)

Match_Stats & Match_Infos

In [ ]:
#fbref_matchsquad = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/output-basic1/fbref_matchsquad.csv/part-00000-701e4a9b-5a6b-4171-9c82-9857304c7151-c000.csv",
#                                 header=True, inferSchema=True)
#fbref_matchinfos = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/output-basic1/fbref_matchinfos.csv/part-00000-38c10226-dab2-4dd9-9b19-1f846b2c2543-c000.csv",
#                                 header=True, inferSchema=True)

In [ ]:
display(fbref_matchstats.limit(5).toPandas())

,Match_Id,Team,Is_Home_Team,Manager,Captain,Formation,Possession,Fouls,Corners,Crosses,...,GCA,Passes_Cmp,Passes_Att,Passes_Cmp_Percentage,Passes_PrgP,Carries,Carries_PrgC,Take_Ons_Att,Take_Ons_Succ,Score
0,39b738ac,Saint-Étienne,Yes,Unknown,Unknown,4-2-3-1,59,14,7,33,...,3,471,588,80.1,54,360,24,29,14,2
1,39b738ac,Strasbourg,No,Unknown,Unknown,4-1-2-1-2,41,10,4,14,...,4,293,410,71.5,23,297,18,12,9,2
2,ed0f2e13,Lille,Yes,Unknown,Unknown,4-2-3-1,50,11,3,12,...,6,408,514,79.4,41,360,24,29,17,3
3,ed0f2e13,Metz,No,Unknown,Unknown,4-2-3-1,50,12,5,14,...,2,378,511,74,38,338,15,23,12,1
4,de5b03fe,Nice,Yes,Unknown,Unknown,4/3/2003,75,14,11,34,...,0,730,826,88.4,69,527,41,40,18,0


In [ ]:
display(fbref_matchinfos.limit(5).toPandas())

,Match_Id,League,Season,Match_Week,Home_Team,Away_Team,Match_Date,Venue_Day,Venue_Time,Attendance,Stadium,Officials,Link
0,a68e623d,Ligue 1,2018,1,Monaco,Toulouse,2017-08-04,Fri,2024-07-03 20:45:00,13572,Stade Louis II,Clément Turpin,https://fbref.com/en/matches/a68e623d/Monaco-T...
1,37f2c25f,Ligue 1,2018,1,Paris S-G,Amiens,2017-08-05,Sat,2024-07-03 17:15:00,46898,Parc des Princes,Mikael Lesage,https://fbref.com/en/matches/37f2c25f/Paris-Sa...
2,4d28b63b,Ligue 1,2018,1,Lyon,Strasbourg,2017-08-05,Sat,2024-07-03 20:00:00,1979,Groupama Stadium,Ruddy Buquet,https://fbref.com/en/matches/4d28b63b/Lyon-Str...
3,68b9eea2,Ligue 1,2018,1,Saint-Étienne,Nice,2017-08-05,Sat,2024-07-03 20:00:00,25879,Stade Geoffroy-Guichard,François Letexier,https://fbref.com/en/matches/68b9eea2/Saint-Et...
4,7d2eb66d,Ligue 1,2018,1,Metz,Guingamp,2017-08-05,Sat,2024-07-03 20:00:00,14595,Stade Saint-Symphorien,Jérôme Miguelgorry,https://fbref.com/en/matches/7d2eb66d/Metz-Gui...


1 row 1 match contains Home and Away

In [ ]:
fbref_matchstats_new = fbref_matchstats.join(fbref_matchinfos, on='Match_Id', how='inner')

In [ ]:
fbref_matchstats_new.count()

25238

In [ ]:
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Manager", lit("Unknown"))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Captain", lit("Unknown"))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Manager", lit("Unknown"))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Captain", lit("Unknown"))

fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Formation",
                                                       when(col("Is_Home_Team") == "Yes", col("Formation")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Formation",
                                                       when(col("Is_Home_Team") == "No", col("Formation")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Possession",
                                                       when(col("Is_Home_Team") == "Yes", col("Possession")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Possession",
                                                       when(col("Is_Home_Team") == "No", col("Possession")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Fouls",
                                                       when(col("Is_Home_Team") == "Yes", col("Fouls")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Fouls",
                                                       when(col("Is_Home_Team") == "No", col("Fouls")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Corners",
                                                       when(col("Is_Home_Team") == "Yes", col("Corners")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Corners",
                                                       when(col("Is_Home_Team") == "No", col("Corners")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Crosses",
                                                       when(col("Is_Home_Team") == "Yes", col("Crosses")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Crosses",
                                                       when(col("Is_Home_Team") == "No", col("Crosses")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Aerials_Won",
                                                       when(col("Is_Home_Team") == "Yes", col("Aerials_Won")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Aerials_Won",
                                                       when(col("Is_Home_Team") == "No", col("Aerials_Won")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Clearances",
                                                       when(col("Is_Home_Team") == "Yes", col("Clearances")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Clearances",
                                                       when(col("Is_Home_Team") == "No", col("Clearances")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Offsides",
                                                       when(col("Is_Home_Team") == "Yes", col("Offsides")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Offsides",
                                                       when(col("Is_Home_Team") == "No", col("Offsides")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Goal_Kicks",
                                                       when(col("Is_Home_Team") == "Yes", col("Goal_Kicks")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Goal_Kicks",
                                                       when(col("Is_Home_Team") == "No", col("Goal_Kicks")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Throw_Ins",
                                                       when(col("Is_Home_Team") == "Yes", col("Throw_Ins")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Throw_Ins",
                                                       when(col("Is_Home_Team") == "No", col("Throw_Ins")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Long_Balls",
                                                       when(col("Is_Home_Team") == "Yes", col("Long_Balls")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Long_Balls",
                                                       when(col("Is_Home_Team") == "No", col("Long_Balls")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Total_Players_Stats", lit("Unknown"))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Total_Players_Stats", lit("Unknown"))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Minutes",
                                                       when(col("Is_Home_Team") == "Yes", col("Minutes")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Minutes",
                                                       when(col("Is_Home_Team") == "No", col("Minutes")))

In [ ]:
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Gls",
                                                       when(col("Is_Home_Team") == "Yes", col("Gls")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Gls",
                                                       when(col("Is_Home_Team") == "No", col("Gls")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Ast",
                                                       when(col("Is_Home_Team") == "Yes", col("Ast")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Ast",
                                                       when(col("Is_Home_Team") == "No", col("Ast")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_PK",
                                                       when(col("Is_Home_Team") == "Yes", col("PK")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_PK",
                                                       when(col("Is_Home_Team") == "No", col("PK")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_PK_Att",
                                                       when(col("Is_Home_Team") == "Yes", col("PK_Att")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_PK_Att",
                                                       when(col("Is_Home_Team") == "No", col("PK_Att")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Sh",
                                                       when(col("Is_Home_Team") == "Yes", col("Sh")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Sh",
                                                       when(col("Is_Home_Team") == "No", col("Sh")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_SoT",
                                                       when(col("Is_Home_Team") == "Yes", col("SoT")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_SoT",
                                                       when(col("Is_Home_Team") == "No", col("SoT")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_CrdY",
                                                       when(col("Is_Home_Team") == "Yes", col("CrdY")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_CrdY",
                                                       when(col("Is_Home_Team") == "No", col("CrdY")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_CrdR",
                                                       when(col("Is_Home_Team") == "Yes", col("CrdR")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_CrdR",
                                                       when(col("Is_Home_Team") == "No", col("CrdR")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Touches",
                                                       when(col("Is_Home_Team") == "Yes", col("Touches")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Touches",
                                                       when(col("Is_Home_Team") == "No", col("Touches")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Tkl",
                                                       when(col("Is_Home_Team") == "Yes", col("Tkl")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Tkl",
                                                       when(col("Is_Home_Team") == "No", col("Tkl")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Int",
                                                       when(col("Is_Home_Team") == "Yes", col("Int")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Int",
                                                       when(col("Is_Home_Team") == "No", col("Int")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Blocks",
                                                       when(col("Is_Home_Team") == "Yes", col("Blocks")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Blocks",
                                                       when(col("Is_Home_Team") == "No", col("Blocks")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_xG",
                                                       when(col("Is_Home_Team") == "Yes", col("xG")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_xG",
                                                       when(col("Is_Home_Team") == "No", col("xG")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_npxG",
                                                       when(col("Is_Home_Team") == "Yes", col("npxG")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_npxG",
                                                       when(col("Is_Home_Team") == "No", col("npxG")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_xAG",
                                                       when(col("Is_Home_Team") == "Yes", col("xAG")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_xAG",
                                                       when(col("Is_Home_Team") == "No", col("xAG")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_SCA",
                                                       when(col("Is_Home_Team") == "Yes", col("SCA")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_SCA",
                                                       when(col("Is_Home_Team") == "No", col("SCA")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_GCA",
                                                       when(col("Is_Home_Team") == "Yes", col("GCA")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_GCA",
                                                       when(col("Is_Home_Team") == "No", col("GCA")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Cmp_Passes",
                                                       when(col("Is_Home_Team") == "Yes", col("Passes_Cmp")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Cmp_Passes",
                                                       when(col("Is_Home_Team") == "No", col("Passes_Cmp")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Att_Passes",
                                                       when(col("Is_Home_Team") == "Yes", col("Passes_Att")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Att_Passes",
                                                       when(col("Is_Home_Team") == "No", col("Passes_Att")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Cmp_percent_Passes",
                                                       when(col("Is_Home_Team") == "Yes", col("Passes_Cmp_Percentage")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Cmp_percent_Passes",
                                                       when(col("Is_Home_Team") == "No", col("Passes_Cmp_Percentage")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_PrgP_Passes",
                                                       when(col("Is_Home_Team") == "Yes", col("Passes_PrgP")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_PrgP_Passes",
                                                       when(col("Is_Home_Team") == "No", col("Passes_PrgP")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Carries_Carries",
                                                       when(col("Is_Home_Team") == "Yes", col("Carries")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Carries_Carries",
                                                       when(col("Is_Home_Team") == "No", col("Carries")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_PrgC_Carries",
                                                       when(col("Is_Home_Team") == "Yes", col("Carries_PrgC")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_PrgC_Carries",
                                                       when(col("Is_Home_Team") == "No", col("Carries_PrgC")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Att_Take_Ons",
                                                       when(col("Is_Home_Team") == "Yes", col("Take_Ons_Att")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Att_Take_Ons",
                                                       when(col("Is_Home_Team") == "No", col("Take_Ons_Att")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Succ_Take_Ons",
                                                       when(col("Is_Home_Team") == "Yes", col("Take_Ons_Succ")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Succ_Take_Ons",
                                                       when(col("Is_Home_Team") == "No", col("Take_Ons_Succ")))



In [ ]:
fbref_matchstats_new = fbref_matchstats_new.withColumn("Home_Score",
                                                       when(col("Is_Home_Team") == "Yes", col("Score")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Away_Score",
                                                       when(col("Is_Home_Team") == "No", col("Score")))
fbref_matchstats_new = fbref_matchstats_new.withColumn("Row_Id", monotonically_increasing_id())

In [ ]:
fbref_matchstats_new = fbref_matchstats_new.select('Match_Id', 'Match_Date',
                                'Home_Team', 'Home_Manager', 'Home_Captain',
                                'Home_Formation', 'Home_Possession', 'Home_Fouls', 'Home_Corners', 'Home_Crosses',
                                'Home_Aerials_Won', 'Home_Clearances', 'Home_Offsides', 'Home_Goal_Kicks',
                                'Home_Throw_Ins', 'Home_Long_Balls', 'Home_Total_Players_Stats', 'Home_Minutes',
                                'Home_Gls', 'Home_Ast', 'Home_PK', 'Home_PK_Att',
                                'Home_Sh','Home_SoT', 'Home_CrdY', 'Home_CrdR',
                                'Home_Touches','Home_Tkl','Home_Int','Home_Blocks',
                                'Home_xG','Home_npxG', 'Home_xAG',
                                'Home_SCA', 'Home_GCA',
                                'Home_Cmp_Passes', 'Home_Att_Passes', 'Home_Cmp_percent_Passes', 'Home_PrgP_Passes',
                                'Home_Carries_Carries', 'Home_PrgC_Carries',
                                'Home_Att_Take_Ons', 'Home_Succ_Take_Ons',
                                'Away_Team', 'Away_Manager', 'Away_Captain',
                                'Away_Formation', 'Away_Possession', 'Away_Fouls', 'Away_Corners', 'Away_Crosses',
                                'Away_Aerials_Won', 'Away_Clearances', 'Away_Offsides', 'Away_Goal_Kicks',
                                'Away_Throw_Ins', 'Away_Long_Balls', 'Away_Total_Players_Stats', 'Away_Minutes',
                                'Away_Gls', 'Away_Ast', 'Away_PK', 'Away_PK_Att',
                                'Away_Sh','Away_SoT', 'Away_CrdY', 'Away_CrdR',
                                'Away_Touches','Away_Tkl','Away_Int','Away_Blocks',
                                'Away_xG','Away_npxG', 'Away_xAG',
                                'Away_SCA', 'Away_GCA',
                                'Away_Cmp_Passes', 'Away_Att_Passes', 'Away_Cmp_percent_Passes', 'Away_PrgP_Passes',
                                'Away_Carries_Carries', 'Away_PrgC_Carries',
                                'Away_Att_Take_Ons', 'Away_Succ_Take_Ons',
                                'Home_Score','Away_Score', 'Row_Id')

In [ ]:
display(fbref_matchstats_new.limit(5).toPandas())

,Match_Id,Match_Date,Home_Team,Home_Manager,Home_Captain,Home_Formation,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,...,Away_Att_Passes,Away_Cmp_percent_Passes,Away_PrgP_Passes,Away_Carries_Carries,Away_PrgC_Carries,Away_Att_Take_Ons,Away_Succ_Take_Ons,Home_Score,Away_Score,Row_Id
0,39b738ac,2017-11-24,Saint-Étienne,Unknown,Unknown,4-2-3-1,59.0,14.0,7.0,33.0,...,None,None,None,None,None,None,None,2,None,0
1,39b738ac,2017-11-24,Saint-Étienne,Unknown,Unknown,None,NaN,NaN,NaN,NaN,...,410,71.5,23,297,18,12,9,None,2,1
2,ed0f2e13,2018-04-28,Lille,Unknown,Unknown,4-2-3-1,50.0,11.0,3.0,12.0,...,None,None,None,None,None,None,None,3,None,2
3,ed0f2e13,2018-04-28,Lille,Unknown,Unknown,None,NaN,NaN,NaN,NaN,...,511,74,38,338,15,23,12,None,1,3
4,de5b03fe,2018-02-03,Nice,Unknown,Unknown,4/3/2003,75.0,14.0,11.0,34.0,...,None,None,None,None,None,None,None,0,None,4


In [ ]:
match_home = fbref_matchstats_new.filter(col("Row_Id") % 2 == 0)
match_away = fbref_matchstats_new.filter(col("Row_Id") % 2 != 0)

In [ ]:
match_home.count()

12619

In [ ]:
match_away.count()

12619

In [ ]:
columns_to_drop_home = ['Away_Manager', 'Away_Captain',
'Away_Formation', 'Away_Possession', 'Away_Fouls', 'Away_Corners', 'Away_Crosses',
'Away_Aerials_Won', 'Away_Clearances', 'Away_Offsides', 'Away_Goal_Kicks',
'Away_Throw_Ins', 'Away_Long_Balls', 'Away_Total_Players_Stats', 'Away_Minutes',
'Away_Gls', 'Away_Ast', 'Away_PK', 'Away_PK_Att',
'Away_Sh','Away_SoT', 'Away_CrdY', 'Away_CrdR',
'Away_Touches','Away_Tkl','Away_Int','Away_Blocks',
'Away_xG','Away_npxG', 'Away_xAG',
'Away_SCA', 'Away_GCA',
'Away_Cmp_Passes', 'Away_Att_Passes', 'Away_Cmp_percent_Passes', 'Away_PrgP_Passes',
'Away_Carries_Carries', 'Away_PrgC_Carries',
'Away_Att_Take_Ons', 'Away_Succ_Take_Ons', 'Away_Score', 'Row_Id']


match_home = match_home.select([column for column in match_home.columns if column not in columns_to_drop_home])

columns_to_drop_away = ['Home_Manager', 'Home_Captain',
'Home_Formation', 'Home_Possession', 'Home_Fouls', 'Home_Corners', 'Home_Crosses',
'Home_Aerials_Won', 'Home_Clearances', 'Home_Offsides', 'Home_Goal_Kicks',
'Home_Throw_Ins', 'Home_Long_Balls', 'Home_Total_Players_Stats', 'Home_Minutes',
'Home_Gls', 'Home_Ast', 'Home_PK', 'Home_PK_Att',
'Home_Sh','Home_SoT', 'Home_CrdY', 'Home_CrdR',
'Home_Touches','Home_Tkl','Home_Int','Home_Blocks',
'Home_xG','Home_npxG', 'Home_xAG',
'Home_SCA', 'Home_GCA',
'Home_Cmp_Passes', 'Home_Att_Passes', 'Home_Cmp_percent_Passes', 'Home_PrgP_Passes',
'Home_Carries_Carries', 'Home_PrgC_Carries',
'Home_Att_Take_Ons', 'Home_Succ_Take_Ons', 'Home_Score', 'Row_Id']

match_away = match_away.select([column for column in match_away.columns if column not in columns_to_drop_away])

In [ ]:
display(match_away.limit(5).toPandas())

,Match_Id,Match_Date,Home_Team,Away_Team,Away_Manager,Away_Captain,Away_Formation,Away_Possession,Away_Fouls,Away_Corners,...,Away_GCA,Away_Cmp_Passes,Away_Att_Passes,Away_Cmp_percent_Passes,Away_PrgP_Passes,Away_Carries_Carries,Away_PrgC_Carries,Away_Att_Take_Ons,Away_Succ_Take_Ons,Away_Score
0,39b738ac,2017-11-24,Saint-Étienne,Strasbourg,Unknown,Unknown,4-1-2-1-2,41,10,4,...,4,293,410,71.5,23,297,18,12,9,2
1,ed0f2e13,2018-04-28,Lille,Metz,Unknown,Unknown,4-2-3-1,50,12,5,...,2,378,511,74,38,338,15,23,12,1
2,de5b03fe,2018-02-03,Nice,Toulouse,Unknown,Unknown,4/3/2003,25,14,1,...,2,204,279,73.1,26,246,17,21,13,1
3,43ebed63,2017-12-20,Guingamp,Saint-Étienne,Unknown,Unknown,4/3/2003,20,13,0,...,2,90,179,50.3,11,73,5,7,3,1
4,1aef5397,2017-08-12,Caen,Saint-Étienne,Unknown,Unknown,4/3/2003,53,15,2,...,2,340,454,74.9,32,260,16,16,10,1


In [ ]:
combine_match = match_home.join(match_away, on=['Match_Id', 'Match_Date', 'Home_Team', 'Away_Team'], how='inner')

In [ ]:
combine_match.count()

12649

In [ ]:
combine_match = combine_match.select(match_home['Match_Id'], match_home['Match_Date'],
                                match_home['Home_Team'], 'Home_Manager', 'Home_Captain',
                                'Home_Formation', 'Home_Possession', 'Home_Fouls', 'Home_Corners', 'Home_Crosses',
                                'Home_Aerials_Won', 'Home_Clearances', 'Home_Offsides', 'Home_Goal_Kicks',
                                'Home_Throw_Ins', 'Home_Long_Balls', 'Home_Total_Players_Stats', 'Home_Minutes',
                                'Home_Gls', 'Home_Ast', 'Home_PK', 'Home_PK_Att',
                                'Home_Sh','Home_SoT', 'Home_CrdY', 'Home_CrdR',
                                'Home_Touches','Home_Tkl','Home_Int','Home_Blocks',
                                'Home_xG','Home_npxG', 'Home_xAG',
                                'Home_SCA', 'Home_GCA',
                                'Home_Cmp_Passes', 'Home_Att_Passes', 'Home_Cmp_percent_Passes', 'Home_PrgP_Passes',
                                'Home_Carries_Carries', 'Home_PrgC_Carries',
                                'Home_Att_Take_Ons', 'Home_Succ_Take_Ons',
                                match_home['Away_Team'], 'Away_Manager', 'Away_Captain',
                                'Away_Formation', 'Away_Possession', 'Away_Fouls', 'Away_Corners', 'Away_Crosses',
                                'Away_Aerials_Won', 'Away_Clearances', 'Away_Offsides', 'Away_Goal_Kicks',
                                'Away_Throw_Ins', 'Away_Long_Balls', 'Away_Total_Players_Stats', 'Away_Minutes',
                                'Away_Gls', 'Away_Ast', 'Away_PK', 'Away_PK_Att',
                                'Away_Sh','Away_SoT', 'Away_CrdY', 'Away_CrdR',
                                'Away_Touches','Away_Tkl','Away_Int','Away_Blocks',
                                'Away_xG','Away_npxG', 'Away_xAG',
                                'Away_SCA', 'Away_GCA',
                                'Away_Cmp_Passes', 'Away_Att_Passes', 'Away_Cmp_percent_Passes', 'Away_PrgP_Passes',
                                'Away_Carries_Carries', 'Away_PrgC_Carries',
                                'Away_Att_Take_Ons', 'Away_Succ_Take_Ons',
                                'Home_Score','Away_Score')

In [ ]:
display(combine_match.limit(5).toPandas())

,Match_Id,Match_Date,Home_Team,Home_Manager,Home_Captain,Home_Formation,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,...,Away_Cmp_Passes,Away_Att_Passes,Away_Cmp_percent_Passes,Away_PrgP_Passes,Away_Carries_Carries,Away_PrgC_Carries,Away_Att_Take_Ons,Away_Succ_Take_Ons,Home_Score,Away_Score
0,f54cc2e1,2018-12-01,Manchester City,Unknown,Unknown,4/3/2003,72,9,8,17,...,225,321,70.1,23,211,10,14,8,3,1
1,78fdbaf6,2018-11-11,Manchester City,Unknown,Unknown,4/3/2003,65,12,5,25,...,335,409,81.9,26,219,10,11,4,3,1
2,f74a4680,2018-09-01,Manchester City,Unknown,Unknown,4-1-3-2,78,5,4,21,...,145,231,62.8,12,130,6,9,7,2,1
3,ac4523c2,2019-11-02,West Ham,Unknown,Unknown,4-1-4-1,70,10,7,34,...,185,285,64.9,22,201,18,24,16,2,3
4,928467bd,2019-08-09,Liverpool,Unknown,Unknown,4/3/2003,57,9,11,32,...,307,404,76.0,26,302,12,16,10,4,1


Export to /output-basic

In [ ]:
#fbreb
fbref_matchinfos.coalesce(1).write.option("header", "true").csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchinfos.csv")
fbref_matchsquad.coalesce(1).write.option("header", "true").csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchsquad.csv")
fbref_matchgoals.coalesce(1).write.option("header", "true").csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchgoals.csv")
fbref_matchstats.coalesce(1).write.option("header", "true").csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchstats.csv")
fbref_matchplayerstats.coalesce(1).write.option("header", "true").csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchplayerstats.csv")

#sofifa
sofifa_players_attr.coalesce(1).write.option("header", "true").csv("gs://football_data_etl/football_data_extracted/output-basic/sofifa_players_attr.csv")

#dataset_model
combine_match.coalesce(1).write.option("header", "true").csv("gs://football_data_etl/football_data_extracted/output-basic/matchs_dataset_model.csv")